<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **5. Wi-Fi Contention Protocol**


<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 5.1 Automatic Repeat reQuest (ARQ) Protocols

Automatic Repeat reQuest (ARQ) schemes are error-control protocols used in data transmission systems to ensure reliable communication. They operate by detecting errors in transmitted data and retransmitting lost or corrupted packets. ARQ schemes are essential in networking and communication protocols to handle unreliable transmission media. There are three common ARQ schemes:

In Stop-and-Wait ARQ, the sender sends one packet and waits for an acknowledgment (ACK) before sending the next one. If no ACK is received after a timeout, the packet is retransmitted. This approach is simple but inefficient, as the sender is idle during the round-trip delay.

In Go-Back-N ARQ, the sender can send multiple packets without waiting for an ACK for each one, but if a packet is lost or corrupted, all subsequent packets are retransmitted (even if they were successfully received). This is known to be more efficient than Stop-and-Wait but suffers from retransmission overhead.

The Selective Repeat ARQ lets a sender transmit multiple packets, and the receiver acknowledges each one individually. If a packet is lost, only the lost packet is retransmitted, not the subsequent packets. Compared to the other schemes, this is the most efficient, as it minimizes unnecessary retransmissions, but requires more complex receiver logic.


#### 5.1.1 Simulation of an Automatic Repeat reQuest (ARQ) Protocol

In [ ]:
import matplotlib.pyplot as plt
import random

import numpy as np
import math
from scipy.stats import binom

In [ ]:
NUM_PACKETS = 30000  # Number of packets to send
WINDOW_SIZE = 10    # Window size for Go-Back-N and Selective Repeat
LOSS_PROBS = [i / 100 for i in range(0, 51, 5)]  # Packet loss probabilities from 0 to 50%

def simulate_stop_and_wait(num_packets, loss_prob):
    transmitted = 0
    retransmissions = 0
    successful = 0

    for i in range(num_packets):
        transmitted += 1
        while random.random() < loss_prob:
            retransmissions += 1
            transmitted += 1
        successful += 1

    throughput = successful /  (transmitted+retransmissions*1.2) / 1.4
    return throughput

def simulate_go_back_n(num_packets, window_size, loss_prob):
    transmitted = 0
    retransmissions = 0
    successful = 0
    base = 0
    next_seq = 0

    while successful < num_packets:
        while next_seq < base + window_size and next_seq < num_packets:
            transmitted += 1
            if random.random() < loss_prob:
                retransmissions += next_seq - base
                next_seq = base
            else:
                next_seq += 1
                successful += 1
        base = next_seq

    throughput = successful / (transmitted+retransmissions*1.2)
    return throughput

def simulate_selective_repeat(num_packets, window_size, loss_prob):
    transmitted = 0
    retransmissions = 0
    successful = 0
    base = 0
    acked = [False] * num_packets

    while successful < num_packets:
        for i in range(base, min(base + window_size, num_packets)):
            if not acked[i]:
                transmitted += 1
                if random.random() < loss_prob:
                    retransmissions += 1
                else:
                    acked[i] = True
                    successful += 1
        while base < num_packets and acked[base]:
            base += 1

    throughput = successful / (transmitted+retransmissions*1.2)
    return throughput

# Collect data for plotting
sw_throughputs = []
gbn_throughputs = []
sr_throughputs = []

for prob in LOSS_PROBS:
    sw_throughputs.append(100*simulate_stop_and_wait(NUM_PACKETS, prob))
    gbn_throughputs.append(100*simulate_go_back_n(NUM_PACKETS, WINDOW_SIZE, prob))
    sr_throughputs.append(100*simulate_selective_repeat(NUM_PACKETS, WINDOW_SIZE, prob))

#### 5.1.2 Plot of Simulated Average Throughputs

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
theLineWidth = 2

ax.cla()
ax.axis([None, None, None, None])

plt.plot(LOSS_PROBS, sw_throughputs, label='Stop-and-Wait', linewidth=theLineWidth)
plt.plot(LOSS_PROBS, gbn_throughputs, label='Go-Back-N (Window Size=' + str(WINDOW_SIZE) + ')', linewidth=theLineWidth)    
plt.plot(LOSS_PROBS, sr_throughputs, label='Selective Repeat (Window Size=' + str(WINDOW_SIZE) + ')', linewidth=theLineWidth)
plt.xlabel('Packet Loss Probability')
plt.ylabel('Throughput [%]')
plt.grid(True)
plt.xlim(0, 0.5)
plt.ylim(0, 100)
ax.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper right", facecolor=(.6, 0.61, 0.8))
plt.tight_layout()



### 5.2 Access Protocols

There are three well-known and often used access protocols in packet radio systems: (Pure) ALOHA, Slotted ALOHA, and Carrier Sense Multiple Access. 

#### 5.2.1 ALOHA

ALOHA was first introduced by the University of Hawaii; the abbreviation has no meaning in the context of access protocols. In Hawaiian culture, Aloha means hello or goodbye. It also expresses peace, compassion, kindness, and care for others and nature.

The first ALOHA network, developed in the early 1970s, connected computers across the Hawaiian islands Oahu, Maui, Hawaii, and Kauai. The ALOHA system used radio communication to enable these islands to share data, making it one of the earliest implementations of a wireless network.

ALOHA is a simple channel access protocol designed for wireless communication systems, particularly in shared or broadcast channels. It allows devices to transmit data whenever they have data to send, without any coordination between devices.
 - A device sends data as soon as it is ready, without checking if the channel is busy.
 - After transmitting, the device waits for an acknowledgment (ACK) from the receiver.
 - If an ACK is received, the transmission was successful. If not (due to a collision with another device's transmission), the device waits a random period of time before retransmitting.
   
The main advantage is the implementation simplicity. Disadvantages are the high probability of collision and low efficiency. Since devices transmit whenever they want, multiple devices may send data at the same time, resulting in collisions. The maximum throughput (successful transmission rate) is limited to about 18.4%, meaning a lot of transmission time is wasted on collisions.

#### 5.2.2 Slotted ALOHA
Slotted ALOHA improves upon the ALOHA protocol by dividing time into discrete slots. Devices can only transmit at the beginning of a time slot, reducing the chance of collisions.
 - Time is divided into uniform slots equal to the time it takes to send one data packet.
 - Devices can only transmit data at the start of a time slot.
 - If two or more devices attempt to transmit in the same slot, a collision occurs, and the data is lost.
 - After a collision, each device waits for a random number of slots before retransmitting.
   
Advantages of Slotted ALOHA over Pure ALOHA are the lower collision probability and the higher efficiency. Since transmissions are restricted to time slots, the likelihood of two devices transmitting at exactly the same time is reduced compared to pure ALOHA. Slotted ALOHA can achieve a maximum throughput of around 37%, which is twice that of pure ALOHA. The main disadvantage is the need for time synchronization among the devices. Devices must synchronize to a common time slot structure, which adds complexity.

Slotted ALOHA itself is not used in 5G, but its principles are applied. The original protocol as discussed here is too simple and inefficient for modern networks, with a maximum efficiency of about 37 percent, but the idea of random access in time slots influenced how 5G devices find each other and connect. In 5G, a logical random access channel defines specific time and frequency slots for devices to send a signal when they want to connect. If two or more devices pick the same slot, a collision occurs, and retransmissions with backoff timers are used to resolve it. This makes 5G’s random access a much more advanced and efficient descendant of slotted ALOHA, even though the original protocol itself is not directly used. The concept of backoff timers is also applied in carrier sense multiple access protocols, see below.


#### 5.2.3 CSMA (Carrier Sense Multiple Access)
CSMA (Carrier Sense Multiple Access) is a more sophisticated protocol that reduces the likelihood of collisions by requiring devices to sense or check the status of the channel before transmitting. It operates under the principle of "listen before talk."
 - Before a device transmits, it checks if the channel is busy.
 - If the channel is idle (no one else is transmitting), the device begins transmission.
 - If the channel is busy, the device waits until it becomes idle.
 - If two or more devices sense that the channel is idle at the same time and transmit simultaneously, a collision occurs, which will require a retransmission.

CSMA can be implemented in several variations. CSMA with Collision Detection (CSMA/CD) is used in Ethernet. In CSMA/CD, devices monitor the channel while transmitting to detect if a collision has occurred. If a collision is detected, the transmission stops immediately, and the device waits a random amount of time before attempting to retransmit. The other variation is CSMA with Collision Avoidance (CSMA/CA), which is predominantly used in Wi-Fi (IEEE 802.11). Instead of detecting collisions (which is hard in wireless systems), devices take steps to avoid collisions by using random backoff. After sensing the channel is idle, the device waits a random period before transmitting to avoid simultaneous transmissions. Receivers also send Acknowledgements (ACKs) for successful transmissions. If an ACK is not received, the transmitter assumes a collision occurred and retransmits.

The advantage of CSMA is that it is more efficient than ALOHA. By sensing the channel and waiting for it to be idle, CSMA reduces the likelihood of collisions and wasted transmission attempts. Collision detection/avoidance further reduces inefficiencies.

Disadvantages are the delay and complexity. There is still a possibility of collisions if two or more devices sense the channel as idle at the same time due to propagation delay (the time it takes for a signal to travel through the network). CSMA is more complex than ALOHA because of backoff or retransmission strategies. CSMA requires devices to monitor the channel.


#### 5.2.4 Comparison of ALOHA, Slotted ALOHA, and CSMA:

| **Protocol**    | **Channel Access Method**                     | **Collision Handling**      | **Efficiency**  | **Applications**                 |
|-----------------|-----------------------------------------------|-----------------------------|-----------------|-----------------------------------|
| **ALOHA**       | Transmit whenever data is ready               | Retransmit after collisions  | Low (~18.4%)    | Early wireless systems, satellites|
| **Slotted ALOHA** | Transmit at the beginning of time slots       | Retransmit after collisions  | Moderate (~37%) | TDMA systems, RFID               |
| **CSMA**        | Sense channel before transmitting             | Collision detection or avoidance (CD/CA) | High (better than ALOHA) | Ethernet (CSMA/CD), Wi-Fi (CSMA/CA)|


#### 5.2.5 Simulation of Access Protocols

In [ ]:
def slotted_aloha(arrival_times: np.ndarray):
    count_arrivals = np.zeros(math.ceil(arrival_times[-1]) + 1)
    for val in arrival_times:
        count_arrivals[math.ceil(val)] += 1
    return np.sum(1 * (count_arrivals == 1))


def pure_aloha(inter_arrival_time: np.ndarray):
    success = 0
    for j in range(len(inter_arrival_time)):
        if j - 1 >= 0 and inter_arrival_time[j - 1] >= 1 and j + 1 < N and inter_arrival_time[j + 1] >= 1:
            success += 1
    return success


def csma_generalized(arrival_times: np.ndarray, p, cd: bool = False):
    beta = .1
    success = 0
    transmission_ends = 0
    collision_detect_possible = 0
    successfully_sending = False
    pending_packets = 0
    for j, time in enumerate(arrival_times):
        if time < collision_detect_possible:
            # channel is busy, but detected as free due to beta delay in detection of ongoing transmission
            if cd:
                transmission_ends = min(transmission_ends, time + beta)
            else:
                transmission_ends = time + 1
            if successfully_sending:
                success -= 1
                successfully_sending = False
        elif time < transmission_ends:
            # channel is busy, do nothing
            pending_packets += 1
            if (j + 1 < len(arrival_times) and arrival_times[j + 1] >= transmission_ends) \
                    or j + 1 == len(arrival_times):
                if p == 1:
                    x = pending_packets
                elif p == 0:
                    x = 0
                else:
                    x = binom(pending_packets, p).rvs()
                if x == 1:
                    success += 1
                    successfully_sending = True
                    collision_detect_possible = transmission_ends + beta
                    transmission_ends = transmission_ends + 1
                elif x > 1:
                    collision_detect_possible = transmission_ends + beta
                    transmission_ends = transmission_ends + 1
                pending_packets = 0
        else:
            # channel is free
            successfully_sending = True
            success += 1
            collision_detect_possible = time + beta
            transmission_ends = time + 1
    return success


N = int(10000)
numG = 30
rng = np.random.default_rng()
throughput_pure = np.zeros(numG)
throughput_slotted = np.zeros(numG)
throughput_csma_non_persistent = np.zeros(numG)
throughput_1persistent = np.zeros(numG)
throughput_pt1persistent = np.zeros(numG)
throughput_pt01persistent = np.zeros(numG)
throughput_pt5persistent = np.zeros(numG)
throughput_csma_cd = np.zeros(numG)
G = np.logspace(-1.5, .9, numG)

for i, lam in enumerate(G):
    inter_arrival_time = rng.exponential(1 / lam, N)
    arrival_times = np.array(inter_arrival_time)
    for j in range(1, len(arrival_times)):
        arrival_times[j] += arrival_times[j - 1]
    throughput_pure[i] = pure_aloha(
        inter_arrival_time) / math.ceil(arrival_times[-1])
    throughput_slotted[i] = slotted_aloha(
        arrival_times) / math.ceil(arrival_times[-1])
    throughput_csma_non_persistent[i] = csma_generalized(
        arrival_times, 0) / math.ceil(arrival_times[-1])
    throughput_1persistent[i] = csma_generalized(
        arrival_times, 1) / math.ceil(arrival_times[-1])
    throughput_pt1persistent[i] = csma_generalized(
        arrival_times, .1) / math.ceil(arrival_times[-1])
    throughput_pt01persistent[i] = csma_generalized(
        arrival_times, .01) / math.ceil(arrival_times[-1])
    throughput_pt5persistent[i] = csma_generalized(
        arrival_times, .5) / math.ceil(arrival_times[-1])
    throughput_csma_cd[i] = csma_generalized(
        arrival_times, 1, True) / math.ceil(arrival_times[-1])

#### 5.2.6 Plot of Average Througputs of Access Protocols

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
theLineWidth = 2

ax.cla()
ax.axis([None, None, None, None])

ax.plot(G, 100*throughput_pure,
        linewidth=theLineWidth+2, label='Pure Aloha')
ax.plot(G, 100*throughput_slotted,
        linewidth=theLineWidth+2, label='Slotted Aloha')
ax.plot(G, 100*throughput_csma_non_persistent,
        linewidth=theLineWidth, label="CSMA non-persistent")
ax.plot(G, 100*throughput_1persistent,
        linewidth=theLineWidth, label="CSMA 1-persistent")
ax.plot(G, 100*throughput_pt1persistent,
        linewidth=theLineWidth, label="CSMA .1-persistent")
ax.plot(G, 100*throughput_pt5persistent,
        linewidth=theLineWidth, label="CSMA .5-persistent")
ax.plot(G, 100*throughput_pt01persistent,
        linewidth=theLineWidth, label="CSMA .01-persistent")
ax.plot(G, 100*throughput_csma_cd,
        linewidth=theLineWidth, label="CSMA Collision Detection")

plt.grid(True)

ax.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper right", facecolor=(.6, 0.61, 0.8))

plt.ylim(0, 100)

ax.set_xlabel('G (attempt rate)')
ax.set_ylabel('Throughput [%]')

plt.tight_layout()

In Slotted ALOHA like in all other protocols, the symbol G is the average number of transmission attempts per slot time, including both new packets and retransmissions. It is a measure of the traffic load offered to the channel. If G = 1, it means on average one packet attempt per slot. If G = 0.5, it means half a packet attempt per slot. Finally, if G > 1, multiple devices are often trying to transmit in the same slot, so collisions rise quickly.

For Slotted ALOHA specifically, the throughput **S** (the rate of successful transmissions) is related to **G** by the formula:

$$
S = G \cdot e^{-G}
$$

The graph above shows this throughput as a percentage of the theoretical maximum. The probability of a successful transmission in a given time interval is $e^{-G}$, and the throughput is the attempt rate times that success probability. The maximum throughput of Slotted ALOHA occurs at $G = 1$, giving $S = \frac{1}{e} \approx 0.37$ packets per slot.

----
### 5.3 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**Explain how Stop-and-Wait performs compared to the alternatives. Why is it still used in practice?** Answer:

(add your answer here)

**What is the maximum throughput as a percentage of the channel capacity of ALOHA and of Slotted ALOHA? Explain the difference.** Answer:

(add your answer here)

**What is the difference between Collision Detection (CD) and Collision Avoidance (CA)? Why is CSMA/CD used in Ethernet, but not in wireless systems like Wi-Fi (which uses Collision Avoidance instead of Detection)?** Answer:

(add your answer here)

**A sender transmits ten packets, but packet 4 is lost while the later packets are received. What is retransmitted under Go-Back-N and under Selective Repeat? Which protocol uses the channel more efficiently, and what additional complexity does it require?** Answer:

(add your answer here)


**Hidden-Terminal Problem: Consider three wireless devices: A and C can both communicate with receiver B, but A and C are too far apart to detect each other’s transmissions. If A is already transmitting to B, C may still sense the channel as idle and begin transmitting. What happens at B, and why does this demonstrate a limitation of CSMA’s “listen before talk” approach?** Answer:

(add your answer here)


**Did you read through and understand all the tasks in this notebook?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?** Answer:

 (please help the team by telling us what part was not understood, and we will work on it)